In [ ]:
# 전처리된 음성데이터로 학습을 진행합니다.
# 학습된 모델은 '비트메이트_TP01/qwen3_ft_output'에 저장됩니다.
# 학습시에는 자원을 많이 먹습니다. A100 추천

In [ ]:
# 비트메이트_TP01/data/dataset/wav_jhc100
speaker = 'jhc100'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -U qwen-tts huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 8.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 146.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 135.1 MB/s eta 0:00:00
  Created wheel for sox: filename=sox-1.5.0-py3-none-any.whl size=40036 sha256=8ca8ae1b409cae42b2da10807461cef14ee854cfdcc9b73eb4da7ac473fa84bd
  Stored in directory: /root/.cache/pip/wheels/8c/c7/e7/baea1f7e79b9eb53addc81cc9b827424f4a7d8c9cc18c03659
Successfully built sox
  Attempting uninstall: huggingface_hub
  

In [ ]:
import os

if not os.path.exists("/content/Qwen3-TTS"):
    !git clone https://github.com/QwenLM/Qwen3-TTS.git

%cd /content/Qwen3-TTS/finetuning

Cloning into 'Qwen3-TTS'...
remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 118 (delta 0), reused 0 (delta 0), pack-reused 115 (from 2)
Receiving objects: 100% (118/118), 6.88 MiB | 24.80 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/Qwen3-TTS/finetuning


In [ ]:
from huggingface_hub import snapshot_download

local_model_dir = "/content/drive/MyDrive/비트메이트_TP01/qwen_models/Qwen3-TTS-12Hz-1.7B-Base"

if not os.path.exists(local_model_dir) or len(os.listdir(local_model_dir)) == 0:
    snapshot_download(
        repo_id="Qwen/Qwen3-TTS-12Hz-1.7B-Base",
        local_dir=local_model_dir,
        local_dir_use_symlinks=False
    )

print("로컬 모델 경로:", local_model_dir)

로컬 모델 경로: /content/drive/MyDrive/비트메이트_TP01/qwen_models/Qwen3-TTS-12Hz-1.7B-Base


In [ ]:
# metadata.txt를 모델이 원하는 형식(train_raw.jsonl)으로 변환해줍니다
import os
import json

base_dir = f"/content/drive/MyDrive/비트메이트_TP01/data/dataset/wav_{speaker}"
wavs_dir = os.path.join(base_dir, "wavs")
metadata_path = os.path.join(base_dir, "metadata.txt")
ref_audio = os.path.join(base_dir, "ref.wav")

output_jsonl = "/content/drive/MyDrive/비트메이트_TP01/train_raw.jsonl"

if not os.path.exists(metadata_path):
    raise FileNotFoundError(f"metadata.txt 없음: {metadata_path}")

if not os.path.exists(ref_audio):
    raise FileNotFoundError(f"ref.wav 없음: {ref_audio}")

rows = []
missing_files = []

with open(metadata_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue

        if "|" not in line:
            print(f"[스킵] {i}번째 줄 형식 이상: {line}")
            continue

        fname, text = line.split("|", 1)
        audio_path = os.path.join(wavs_dir, fname)

        if not os.path.exists(audio_path):
            missing_files.append(audio_path)
            continue

        rows.append({
            "audio": audio_path,
            "text": text.strip(),
            "ref_audio": ref_audio
        })

with open(output_jsonl, "w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("저장 완료:", output_jsonl)
print("유효 샘플 수:", len(rows))

if missing_files:
    print("\n없는 파일들:")
    for p in missing_files[:20]:
        print("-", p)

저장 완료: /content/drive/MyDrive/비트메이트_TP01/train_raw.jsonl
유효 샘플 수: 39


In [ ]:
!apt-get update -qq
!apt-get install -y sox libsox-fmt-all

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libao-common libao4 libid3tag0 libmad0 libopencore-amrnb0 libopencore-amrwb0
  libsox-fmt-alsa libsox-fmt-ao libsox-fmt-base libsox-fmt-mp3 libsox-fmt-oss
  libsox-fmt-pulse libsox3 libwavpack1
Suggested packages:
  libaudio2 libsndio6.1
The following NEW packages will be installed:
  libao-common libao4 libid3tag0 libmad0 libopencore-amrnb0 libopencore-amrwb0
  libsox-fmt-all libsox-fmt-alsa libsox-fmt-ao libsox-fmt-base libsox-fmt-mp3
  libsox-fmt-oss libsox-fmt-pulse libsox3 libwavpack1 sox
0 upgraded, 16 newly installed, 0 to remove and 88 not upgraded.
Need to get 800 kB of archives.
After this operation, 2,533 kB of additional disk space will be

In [ ]:
!python prepare_data.py \
  --device cuda:0 \
  --tokenizer_model_path Qwen/Qwen3-TTS-Tokenizer-12Hz \
  --input_jsonl /content/drive/MyDrive/비트메이트_TP01/train_raw.jsonl \
  --output_jsonl /content/drive/MyDrive/비트메이트_TP01/train_with_codes.jsonl

2026-04-21 05:21:53.026699: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-21 05:21:53.045216: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776748913.067888    2303 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776748913.075464    2303 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776748913.094713    2303 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
# 코랩 환경에서 돌리기 위해 수정
file_path = "/content/Qwen3-TTS/finetuning/sft_12hz.py"

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

text = text.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')
text = text.replace("attn_implementation='flash_attention_2'", "attn_implementation='sdpa'")

old = 'accelerator = Accelerator(gradient_accumulation_steps=4, mixed_precision="bf16", log_with="tensorboard")'
new = '''accelerator = Accelerator(
        gradient_accumulation_steps=4,
        mixed_precision="bf16",
        log_with="tensorboard",
        project_dir="/content/drive/MyDrive/비트메이트_TP01/qwen3_logs"
    )'''

if old in text:
    text = text.replace(old, new)

with open(file_path, "w", encoding="utf-8") as f:
    f.write(text)

print("sft_12hz.py 패치 완료")

sft_12hz.py 패치 완료


In [ ]:
!grep -n "attn_implementation" /content/Qwen3-TTS/finetuning/sft_12hz.py
!grep -n "Accelerator(" /content/Qwen3-TTS/finetuning/sft_12hz.py

56:        attn_implementation="sdpa",
44:    accelerator = Accelerator(


In [ ]:
# 실제 학습을 진행하는 과정
model_path = "/content/drive/MyDrive/비트메이트_TP01/qwen_models/Qwen3-TTS-12Hz-1.7B-Base"
output_path = f"/content/drive/MyDrive/비트메이트_TP01/qwen3_ft_output/{speaker}"
train_path = "/content/drive/MyDrive/비트메이트_TP01/train_with_codes.jsonl"

batch_size = 1
lr = 2e-6
epochs = 4

!python sft_12hz.py \
  --init_model_path {model_path} \
  --output_model_path {output_path} \
  --train_jsonl {train_path} \
  --batch_size {batch_size} \
  --lr {lr} \
  --num_epochs {epochs} \
  --speaker_name {speaker}

2026-04-21 05:22:15.311426: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-21 05:22:15.330155: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776748935.352730    2666 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776748935.360194    2666 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776748935.378692    2666 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
# 학습된 모델을 확인
import os

ckpt_root = f"/content/drive/MyDrive/비트메이트_TP01/qwen3_ft_output/{speaker}"
print(os.listdir(ckpt_root))

['checkpoint-epoch-0', 'checkpoint-epoch-1', 'checkpoint-epoch-2', 'checkpoint-epoch-3']


In [ ]:
# 학습된 모델 테스트
import os
import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel
from IPython.display import Audio, display

ckpt_root = f"/content/drive/MyDrive/비트메이트_TP01/qwen3_ft_output/{speaker}"
test_text = "토끼와 거북이는 빠른 토끼가 느린 거북이를 얕보고 경주 도중 방심해 잠이 드는 사이, 꾸준히 쉬지 않고 나아간 거북이가 결국 먼저 도착해 승리한다는 이야기로, 재능이나 속도보다도 끝까지 성실하게 노력하는 태도가 더 중요하다는 교훈을 전한다."

for name in sorted(os.listdir(ckpt_root)):
    if not name.startswith("checkpoint-epoch-"):
        continue

    ckpt_path = os.path.join(ckpt_root, name)

    # 🔥 핵심: 가중치 없는 체크포인트 제외
    if not os.path.exists(os.path.join(ckpt_path, "model.safetensors")):
        print(f"{name} → skip (no weights)")
        continue

    print(f"\n===== {name} =====")

    tts = Qwen3TTSModel.from_pretrained(
        ckpt_path,
        device_map="cuda:0" if torch.cuda.is_available() else "cpu",
        dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    )

    wavs, sr = tts.generate_custom_voice(
        text=test_text,
        speaker=speaker,
    )

    out_path = f"{name}.wav"
    sf.write(out_path, wavs[0], sr)
    display(Audio(out_path))

Output hidden; open in https://colab.research.google.com to view.